In [1]:
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from peft_implementation import apply_lora, get_trainable_parameters
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch
import time

c:\Users\Maga\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load tokenized splits from preprocessing
train_tok = load_from_disk("train_tok")
val_tok   = load_from_disk("val_tok")
test_tok  = load_from_disk("test_tok")

print(f"Train: {len(train_tok)} | Val: {len(val_tok)} | Test: {len(test_tok)}")

Train: 13994 | Val: 2999 | Test: 3000


In [ ]:
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
LABEL2ID = {"positive": 0, "negative": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
BATCH_SIZE = 16
FULL_FT_EPOCHS = 5
PEFT_EPOCHS = 5
FULL_FT_LR = 2e-5
PEFT_LR = 3e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01

Using device: cuda


In [ ]:
# PyTorch DataLoaders
train_dataloader = DataLoader(train_tok, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_tok, shuffle=False, batch_size=BATCH_SIZE * 2)
test_dataloader = DataLoader(test_tok, shuffle=False, batch_size=BATCH_SIZE * 2)

In [ ]:
# Manual PyTorch training loop
def train_model(model, train_dataloader, val_dataloader, optimizer, epochs, label):

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=len(train_dataloader) * epochs
    )

    best_f1    = 0
    best_state = None

    for epoch in range(epochs):

        # Training
        model.train()
        total_loss = 0

        for batch in train_dataloader:
            optimizer.zero_grad()

            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()

            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        # Evaluation
        model.eval()
        total_val_loss = 0
        all_preds  = []
        all_labels = []

        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                total_val_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_train_loss = total_loss / len(train_dataloader)
        avg_val_loss   = total_val_loss / len(val_dataloader)
        acc = accuracy_score(all_labels, all_preds)
        f1  = f1_score(all_labels, all_preds, average="macro")

        print(f"{label} - Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | Accuracy: {acc:.6f} | Macro F1: {f1:.6f}")

        # Track best model by macro F1 (same as HuggingFace load_best_model_at_end)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best checkpoint with Macro F1: {best_f1:.6f}")

    return model

---
## Part 1 — Full Fine-Tuning

In [6]:
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

print(f"Total parameters: {sum(p.numel() for p in full_model.parameters()):,}")

full_optimizer = AdamW(full_model.parameters(), lr=FULL_FT_LR, weight_decay=WEIGHT_DECAY)

full_start = time.time()
full_model = train_model(full_model, train_dataloader, val_dataloader, full_optimizer, FULL_FT_EPOCHS, "Full Fine-Tuning")
full_end = time.time()

print(f"Full Fine-Tuning Training Time: {(full_end - full_start) / 60:.2f} minutes")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8088.16it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 278,045,955
Full Fine-Tuning - Epoch 1 | Train Loss: 0.970633 | Val Loss: 0.913949 | Accuracy: 0.584195 | Macro F1: 0.583579
Full Fine-Tuning - Epoch 2 | Train Loss: 0.838422 | Val Loss: 0.868457 | Accuracy: 0.594198 | Macro F1: 0.591010
Full Fine-Tuning - Epoch 3 | Train Loss: 0.759078 | Val Loss: 0.825420 | Accuracy: 0.623208 | Macro F1: 0.626541
Full Fine-Tuning - Epoch 4 | Train Loss: 0.682037 | Val Loss: 0.861883 | Accuracy: 0.621874 | Macro F1: 0.626368
Full Fine-Tuning - Epoch 5 | Train Loss: 0.612790 | Val Loss: 0.930951 | Accuracy: 0.617206 | Macro F1: 0.618713

Restored best checkpoint with Macro F1: 0.626541
Full Fine-Tuning Training Time: 8.46 minutes


In [7]:
full_model.save_pretrained("./pytorch_full_ft_model")
tokenizer.save_pretrained("./pytorch_full_ft_model")
print("Full fine-tuned model saved.")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]

Full fine-tuned model saved.


---
## Part 2 — PEFT / LoRA Fine-Tuning

In [8]:
peft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Freeze all parameters first before applying LoRA
# (reference: https://github.com/huggingface/peft/blob/main/src/peft/tuners/lora/model.py)
for param in peft_model.parameters():
    param.requires_grad = False

# Apply our manual LoRA implementation — target_modules match XLM-R attention layer names
peft_model = apply_lora(
    peft_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

# Unfreeze classifier head so it can be trained
for param in peft_model.classifier.parameters():
    param.requires_grad = True

peft_model = peft_model.to(device)
get_trainable_parameters(peft_model)

# Only pass trainable parameters to the optimizer
peft_optimizer = AdamW(
    filter(lambda p: p.requires_grad, peft_model.parameters()),
    lr=PEFT_LR,
    weight_decay=WEIGHT_DECAY
)

peft_start = time.time()
peft_model = train_model(peft_model, train_dataloader, val_dataloader, peft_optimizer, PEFT_EPOCHS, "PEFT/LoRA")
peft_end   = time.time()

print(f"PEFT/LoRA Training Time: {(peft_end - peft_start) / 60:.2f} minutes")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11217.00it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 1,182,723 / 278,635,779 (0.42%)
PEFT/LoRA - Epoch 1 | Train Loss: 0.964229 | Val Loss: 0.870041 | Accuracy: 0.604535 | Macro F1: 0.604202
PEFT/LoRA - Epoch 2 | Train Loss: 0.854536 | Val Loss: 0.836978 | Accuracy: 0.612204 | Macro F1: 0.613718
PEFT/LoRA - Epoch 3 | Train Loss: 0.809597 | Val Loss: 0.822399 | Accuracy: 0.627876 | Macro F1: 0.630871
PEFT/LoRA - Epoch 4 | Train Loss: 0.778175 | Val Loss: 0.825103 | Accuracy: 0.629210 | Macro F1: 0.634094
PEFT/LoRA - Epoch 5 | Train Loss: 0.755038 | Val Loss: 0.826300 | Accuracy: 0.630544 | Macro F1: 0.633769

Restored best checkpoint with Macro F1: 0.634094
PEFT/LoRA Training Time: 4.80 minutes


In [ ]:
peft_model.save_pretrained("./pytorch_peft_model")
tokenizer.save_pretrained("./pytorch_peft_model")
print("LoRA fine-tuned model saved.")

---
## Part 3 — Evaluation on Validation Set

In [9]:
def evaluate_on_val(model, model_label):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print()
    print(f"  {model_label} — Validation Set Results")
    print()
    print(classification_report(
        all_labels, all_preds,
        target_names=["Positive", "Negative", "Neutral"]
    ))
    return all_preds, all_labels

In [10]:
full_preds, true_labels = evaluate_on_val(full_model, "Full Fine-Tuning")
peft_preds, _ = evaluate_on_val(peft_model, "PEFT/LoRA")


  Full Fine-Tuning — Validation Set Results

              precision    recall  f1-score   support

    Positive       0.71      0.69      0.70       982
    Negative       0.60      0.69      0.65       890
     Neutral       0.56      0.51      0.53      1127

    accuracy                           0.62      2999
   macro avg       0.63      0.63      0.63      2999
weighted avg       0.62      0.62      0.62      2999


  PEFT/LoRA — Validation Set Results

              precision    recall  f1-score   support

    Positive       0.72      0.68      0.70       982
    Negative       0.63      0.66      0.64       890
     Neutral       0.55      0.57      0.56      1127

    accuracy                           0.63      2999
   macro avg       0.64      0.63      0.63      2999
weighted avg       0.63      0.63      0.63      2999



In [11]:
# Side-by-side comparison summary
full_f1  = f1_score(true_labels, full_preds, average="macro")
peft_f1  = f1_score(true_labels, peft_preds, average="macro")
full_acc = accuracy_score(true_labels, full_preds)
peft_acc = accuracy_score(true_labels, peft_preds)

print("\nComparison Summary (Validation Set)")
print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>10}")
print()
print(f"{'Full Fine-Tuning (XLM-R)':<30} {full_acc:>10.4f} {full_f1:>10.4f}")
print(f"{'PEFT/LoRA (XLM-R)':<30} {peft_acc:>10.4f} {peft_f1:>10.4f}")


Comparison Summary (Validation Set)
Model                            Accuracy   Macro F1

Full Fine-Tuning (XLM-R)           0.6232     0.6265
PEFT/LoRA (XLM-R)                  0.6292     0.6341


---
## Part 4 — Inference on Unlabelled Test Set

In [ ]:
'''
# Generate predictions on unlabelled test set
def predict_on_test(model, model_label):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())

    pred_labels = [ID2LABEL[p] for p in all_preds]

    from collections import Counter
    print(f"\n> {model_label} — Test Set Prediction Distribution")
    print(Counter(pred_labels))
    return pred_labels

full_test_preds = predict_on_test(full_model, "Full Fine-Tuning")
peft_test_preds = predict_on_test(peft_model, "PEFT/LoRA")
'''

In [12]:
# Inference on sample Hinglish sentences
def predict_sentiment(text, model, tokenizer, id2label):
    model.eval()
    inputs = tokenizer(
        text, return_tensors="pt",
        max_length=128, truncation=True, padding="max_length"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id    = torch.argmax(logits, dim=-1).item()
    confidence = torch.softmax(logits, dim=-1).max().item()
    return id2label[pred_id], round(confidence, 4)


sample_sentences = [
    "yaar aaj ka din bahut accha tha", # positive
    "mujhe bilkul pasand nahi yeh cheez", # negative
    "sab kuch theek hai, koi baat nahi", # neutral
]

print("\n> Full Fine-Tuning Inference <")
for s in sample_sentences:
    label, conf = predict_sentiment(s, full_model, tokenizer, ID2LABEL)
    print(f"  '{s}'\n  >> {label} (confidence: {conf})\n")

print("\n> PEFT/LoRA Inference <")
for s in sample_sentences:
    label, conf = predict_sentiment(s, peft_model, tokenizer, ID2LABEL)
    print(f"  '{s}'\n  >> {label} (confidence: {conf})\n")


> Full Fine-Tuning Inference <
  'yaar aaj ka din bahut accha tha'
  >> positive (confidence: 0.5992)

  'mujhe bilkul pasand nahi yeh cheez'
  >> neutral (confidence: 0.7104)

  'sab kuch theek hai, koi baat nahi'
  >> neutral (confidence: 0.7616)


> PEFT/LoRA Inference <
  'yaar aaj ka din bahut accha tha'
  >> positive (confidence: 0.7417)

  'mujhe bilkul pasand nahi yeh cheez'
  >> neutral (confidence: 0.7435)

  'sab kuch theek hai, koi baat nahi'
  >> neutral (confidence: 0.7073)



Same issue as the HuggingFace script!